<a id='1-3'></a>
### 1.3 Using the OpenAI-compatible client directly

Both Together.ai and Ollama expose an OpenAI-compatible endpoint (`/v1`),  
so you can use the `openai` Python library with either — just point it at the right URL.  
Gemini does not support this protocol; the cell below falls back to `utils.py` automatically.


<a id='0'></a>
## 0 — Setup

The `utils.py` in this folder auto-loads your `.env` on import.  
Make sure your `.env` file is at `D:\AI_Engineer-RAG\.env` with `LLM_BACKEND`  
set to your preferred backend (`ollama`, `gemini`, or `together`).

See `utils.py` for the full list of supported env variables.


In [1]:
# ── Import everything we need from utils.py ───────────────────────────────────
# utils.py is in the same folder as this notebook.
# The backend is controlled by LLM_BACKEND in your .env (ollama | gemini | together)
import os
from dotenv import load_dotenv
from utils import (
    generate_with_single_input,
    generate_with_multiple_input,
)

load_dotenv()
backend = os.environ.get('LLM_BACKEND', 'ollama')
print(f'✅ utils.py loaded')
print(f'✅ Active backend: {backend}')

if backend == 'ollama':
    host = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
    model = os.environ.get('OLLAMA_MODEL', 'qwen2.5:7b')
    print(f'   Ollama host : {host}')
    print(f'   Ollama model: {model}')
    print('   Make sure `ollama serve` is running and the model is pulled.')
elif backend == 'gemini':
    key = os.environ.get('GEMINI_API_KEY', '')
    print(f'   GEMINI_API_KEY: {key[:8]}...' if key else '   ❌ GEMINI_API_KEY not found')
elif backend == 'together':
    key = os.environ.get('TOGETHER_API_KEY', '')
    print(f'   TOGETHER_API_KEY: {key[:8]}...' if key else '   ❌ TOGETHER_API_KEY not found')


✅ utils.py loaded
✅ Active backend: ollama
   Ollama host : http://localhost:11434
   Ollama model: qwen3.5:4B
   Make sure `ollama serve` is running and the model is pulled.


---
<a id='1'></a>
## 1 — The Two LLM Functions

All LLM calls in this course go through one of two functions in `utils.py`.  
Understanding them now means you never have to think about API mechanics again.

<a id='1-1'></a>
### 1.1 `generate_with_single_input` — one prompt, one answer

Use this when you have a single question or instruction to send.

**Parameters:**

| Parameter | Type | Default | What it does |
|-----------|------|---------|--------------|
| `prompt` | str | required | The text to send to the model |
| `role` | str | `'user'` | Who is sending — `'user'`, `'system'`, `'assistant'` |
| `max_tokens` | int | 500 | Max length of the response |
| `temperature` | float | None | 0.0 = deterministic, 1.0+ = creative |
| `model` | str | Qwen2.5-7B | Which Together.ai model to use |

**Returns:** `{'role': 'assistant', 'content': '...'}`

In [2]:
# ── Simplest possible call ────────────────────────────────────────────────────
output = generate_with_single_input(
    prompt="What is the capital of France?"
)

# The return is always a dict with 'role' and 'content'
print("Role:   ", output['role'])
print("Content:", output['content'])

Role:    assistant
Content: The capital of France is **Paris**. It is also the country's most populous city and its main center for business, culture, education, and commerce. Located on the Seine River, Paris has been the capital of France for over 800 years and is home to world-renowned landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral.


In [3]:
# ── DEBUG: check utils.py version and test raw call ──────────────────────────
import inspect, utils, requests, os, json
from dotenv import load_dotenv
load_dotenv(override=True)

# 1. Which file is being used?
print('utils.py location:', inspect.getfile(utils))

# 2. Does it have 'think' in _call_ollama?
src = inspect.getsource(utils._call_ollama)
print('Has think=False:', '"think"' in src or "'think'" in src)
print()

# 3. Raw direct call to confirm model returns content
host = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
model = os.environ.get('OLLAMA_MODEL', 'qwen3.5:4B')
print(f'Calling {model} at {host}')
resp = requests.post(f'{host}/api/chat', json={
    'model': model,
    'messages': [{'role': 'user', 'content': 'Say: hello'}],
    'stream': False, 'think': False,
    'options': {'num_predict': 20}
}, timeout=60)
data = resp.json()
print('Raw content:', repr(data['message']['content']))


utils.py location: d:\AI_Engineer-RAG\projects\module_01_RAG_Overview\labs\utils.py
Has think=False: True

Calling qwen3.5:4B at http://localhost:11434
Raw content: 'Hello! How can I help you?'


In [4]:
# ── Controlling the output length ─────────────────────────────────────────────
# max_tokens limits how long the response can be
short_output = generate_with_single_input(
    prompt="Explain what an LLM is.",
    max_tokens=50   # force a very short answer
)
print("Short answer:")
print(short_output['content'])

print()

long_output = generate_with_single_input(
    prompt="Explain what an LLM is.",
    max_tokens=300  # allow a full explanation
)
print("Long answer:")
print(long_output['content'])

Short answer:
An **LLM**, or **Large Language Model**, is a type of artificial intelligence (AI) system designed to understand, generate, and manipulate human language. Unlike older chatbots that relied on predefined scripts or simple keyword matching, LLMs are

Long answer:
An **LLM** (Large Language Model) is a specialized type of **artificial intelligence** designed to understand, generate, and manipulate human language. Unlike earlier AI models that followed rigid sets of rules or performed specific, isolated tasks (like recognizing a specific digit), LLMs are capable of grasping the context, nuance, and complexity of natural language across a wide variety of topics.

Here is a breakdown of what makes an LLM unique:

### 1. How It Works: The "Statistical Pattern" Approach
At its core, an LLM is a **machine learning model** trained on massive datasets of text (books, websites, code, articles, etc.).
*   **Training**: During the training phase, the AI analyzes trillions of words to l

In [5]:
# ── Temperature: deterministic vs creative ────────────────────────────────────
# temperature=0.0 → always the same answer (good for factual RAG)
# temperature=1.0 → different answer each run (good for creative tasks)

print("=== temperature=0.0 (run this twice — same result) ===")
det = generate_with_single_input(
    prompt="Give me a one-word color.",
    temperature=0.0
)
print(det['content'])

print()
print("=== temperature=1.0 (run this twice — likely different) ===")
creative = generate_with_single_input(
    prompt="Give me a one-word color.",
    temperature=1.0
)
print(creative['content'])

=== temperature=0.0 (run this twice — same result) ===
Blue

=== temperature=1.0 (run this twice — likely different) ===
Scarlet


<a id='1-2'></a>
### 1.2 `generate_with_multiple_input` — full conversation

Use this when you need:
- A **system message** (to set the model's behavior)
- **Multi-turn chat** (the model needs to remember earlier messages)
- **Conversation history** in the prompt

You pass a **list of message dicts** — the model sees the whole list at once.

In [6]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# The model uses the full history to answer the final question
messages = [
    {'role': 'user',      'content': 'Hello, who won the FIFA World Cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user',      'content': 'Who was the captain?'},  # ← this is the new question
]

# The model knows the context (France, 2018) from the earlier messages
output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:   ", output['role'])
print("Content:", output['content'])

Role:    assistant
Content: The captain of the France national team at the 2018 FIFA World Cup was **Hakim Ziyech**? Wait, that's incorrect. Let me correct that immediately.

The captain was **Paul Pogba**.

He wore the number 10 jersey and was instrumental in leading France to their first World Cup title, with a notable moment being his goal in the final against Croatia.


In [7]:
# ── With a system message ─────────────────────────────────────────────────────
# The system message shapes how the model behaves — tone, persona, constraints
messages_with_system = [
    {
        'role': 'system',
        'content': 'You are a concise assistant. Answer in one sentence only. No extra explanation.'
    },
    {
        'role': 'user',
        'content': 'What is Retrieval Augmented Generation?'
    }
]

output = generate_with_multiple_input(messages=messages_with_system)
print(output['content'])

Retrieval Augmented Generation (RAG) is a hybrid AI technique that combines retrieval of relevant information from an external database with generative models to enhance the accuracy and context of text outputs.


In [8]:
# ── Building conversation history dynamically ─────────────────────────────────
# Pattern you'll use in multi-turn RAG chatbots

history = [
    {'role': 'system', 'content': 'You are a helpful AI assistant.'}
]

def chat(user_message: str, history: list) -> str:
    """Send a message, append it and the response to history, return the answer."""
    history.append({'role': 'user', 'content': user_message})
    response = generate_with_multiple_input(messages=history, max_tokens=150)
    history.append({'role': 'assistant', 'content': response['content']})
    return response['content']


# Turn 1
answer1 = chat("My name is AbdelRahman. What is RAG?", history)
print("Turn 1:", answer1)
print()

# Turn 2 — the model remembers your name from turn 1
answer2 = chat("What did I just tell you my name was?", history)
print("Turn 2:", answer2)

Turn 1: Hello AbdelRahman!

**RAG** stands for **Retrieval-Augmented Generation**. It is a powerful framework used in artificial intelligence, particularly for Large Language Models (LLMs), to combine the strengths of **generative AI** with external knowledge.

Here is a simple breakdown of how it works:

1.  **Retrieval (The "Memory" Step):**
    When you ask a question, the system first searches a specific database, document repository, or search engine to find relevant information related to your query. It doesn't rely solely on what it was "trained" on during its initial setup.

2.  **Augmented (The "Context" Step):**
    The information found in the

Turn 2: You told me your name is **AbdelRahman**.

I remember that because I am continuing the conversation based on what you shared in your first message! How can I help you today?


<a id='1-3'></a>
### 1.3 Using the OpenAI-compatible client directly

Both Together.ai and Ollama expose an OpenAI-compatible endpoint (`/v1`),  
so you can use the `openai` Python library with either — just point it at the right URL.  
Gemini does not support this protocol; the cell below falls back to `utils.py` automatically.


In [9]:
# ── OpenAI-compatible client (optional, Together.ai only) ────────────────────
# Together.ai exposes an OpenAI-compatible endpoint.
# For Ollama and Gemini, use generate_with_single_input / generate_with_multiple_input instead.
#
# Uncomment the block below ONLY if LLM_BACKEND=together in your .env.

import os
from dotenv import load_dotenv
load_dotenv()

backend = os.environ.get('LLM_BACKEND', 'ollama')

if backend == 'together':
    from openai import OpenAI
    client = OpenAI(
        api_key=os.environ.get('TOGETHER_API_KEY'),
        base_url='https://api.together.xyz/v1',
    )
    model = os.environ.get('TOGETHER_MODEL', 'Qwen/Qwen2.5-7B-Instruct-Turbo')
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': 'What is the OpenAI-compatible API format?'}],
        max_tokens=100,
    )
    print(response.choices[0].message.content)

elif backend == 'ollama':
    # Ollama also has an OpenAI-compatible endpoint at /v1
    from openai import OpenAI
    host = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
    model = os.environ.get('OLLAMA_MODEL', 'qwen2.5:7b')
    client = OpenAI(api_key='ollama', base_url=f'{host}/v1')
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': 'What is the OpenAI-compatible API format?'}],
        max_tokens=100,
    )
    print(response.choices[0].message.content)

elif backend == 'gemini':
    # Gemini does not have an OpenAI-compatible endpoint — use utils.py instead
    print('Gemini backend detected — using generate_with_single_input instead.')
    output = generate_with_single_input('What is the OpenAI-compatible API format?', max_tokens=100)
    print(output['content'])


---
<a id='2'></a>
## 2 — Augmenting Prompts with Data

This is the core idea of RAG in its simplest form:

> Instead of asking the LLM a question and hoping it knows the answer,
> you **give it the relevant data** and ask it to answer from that.

<a id='2-1'></a>
### 2.1 The data — a small house dataset

In the real course this data comes from a vector database.  
Here it's hardcoded so we can focus on the prompt-building pattern.

In [10]:
# ── House dataset — list of dicts ─────────────────────────────────────────────
# This structure (list of dicts) is exactly how retrieved documents look in RAG
house_data = [
    {
        "address":      "123 Maple Street",
        "city":         "Springfield",
        "state":        "IL",
        "zip":          "62701",
        "bedrooms":     3,
        "bathrooms":    2,
        "square_feet":  1500,
        "price":        230000,
        "year_built":   1998
    },
    {
        "address":      "456 Elm Avenue",
        "city":         "Shelbyville",
        "state":        "TN",
        "zip":          "37160",
        "bedrooms":     4,
        "bathrooms":    3,
        "square_feet":  2500,
        "price":        320000,
        "year_built":   2005
    },
    {
        "address":      "789 Oak Drive",
        "city":         "Capital City",
        "state":        "NY",
        "zip":          "10001",
        "bedrooms":     5,
        "bathrooms":    4,
        "square_feet":  3800,
        "price":        750000,
        "year_built":   2018
    },
]

print(f"Dataset loaded: {len(house_data)} houses")
print(f"Keys in each record: {list(house_data[0].keys())}")

Dataset loaded: 3 houses
Keys in each record: ['address', 'city', 'state', 'zip', 'bedrooms', 'bathrooms', 'square_feet', 'price', 'year_built']


<a id='2-2'></a>
### 2.2 Formatting the data into a string

An LLM reads text — not Python dicts.  
The first step is converting the structured data into a readable text block.

In [11]:
# ── Format houses into a readable text block ──────────────────────────────────
def house_info_layout(houses: list) -> str:
    """
    Converts a list of house dicts into a human-readable string.
    This string becomes the 'context' block in the augmented prompt.
    
    The multi-line f-string technique (parentheses across lines) keeps
    the code readable without any line-continuation characters.
    """
    layout = ''
    for i, house in enumerate(houses, 1):
        layout += (
            f"[House {i}] {house['address']}, {house['city']}, {house['state']} {house['zip']} — "
            f"{house['bedrooms']} bed / {house['bathrooms']} bath, "
            f"{house['square_feet']:,} sq ft, "
            f"${house['price']:,}, "
            f"built {house['year_built']}.\n"
        )
    return layout


print(house_info_layout(house_data))

[House 1] 123 Maple Street, Springfield, IL 62701 — 3 bed / 2 bath, 1,500 sq ft, $230,000, built 1998.
[House 2] 456 Elm Avenue, Shelbyville, TN 37160 — 4 bed / 3 bath, 2,500 sq ft, $320,000, built 2005.
[House 3] 789 Oak Drive, Capital City, NY 10001 — 5 bed / 4 bath, 3,800 sq ft, $750,000, built 2018.



<a id='2-3'></a>
### 2.3 Building the augmented prompt

The augmented prompt = the data block + the user's question.  
This is the **Augmentation** step of Retrieve → **Augment** → Generate.

In [12]:
# ── Augmented prompt builder ──────────────────────────────────────────────────
def generate_prompt(query: str, houses: list) -> str:
    """
    Builds a full prompt by combining the house data with the user's query.
    
    Note: 'houses' can be a subset of the full dataset — you might retrieve
    only the most relevant houses for the query, not all of them.
    This is exactly what a vector database retriever does in production RAG.
    """
    context = house_info_layout(houses)
    
    prompt = f"""Use the following house listings to answer the user's query.
Only use the information provided below — do not make up data.

House Listings:
{context}
Query: {query}

Answer:"""
    return prompt


# Preview the full prompt before sending it
sample_query = "What is the most expensive house?"
print(generate_prompt(sample_query, house_data))

Use the following house listings to answer the user's query.
Only use the information provided below — do not make up data.

House Listings:
[House 1] 123 Maple Street, Springfield, IL 62701 — 3 bed / 2 bath, 1,500 sq ft, $230,000, built 1998.
[House 2] 456 Elm Avenue, Shelbyville, TN 37160 — 4 bed / 3 bath, 2,500 sq ft, $320,000, built 2005.
[House 3] 789 Oak Drive, Capital City, NY 10001 — 5 bed / 4 bath, 3,800 sq ft, $750,000, built 2018.

Query: What is the most expensive house?

Answer:


<a id='2-4'></a>
### 2.4 Comparing: without vs with context

This is the central demonstration of Module 1.  
The LLM does not know your specific house data — it was not in its training.  
Watch what happens when you ask with and without providing the data.

In [13]:
# ── Ask WITHOUT context ───────────────────────────────────────────────────────
# The LLM has no idea what houses you are referring to
query = "What is the most expensive house? And the biggest one?"

response_without = generate_with_single_input(
    prompt=query,
    role='user',
    max_tokens=150
)

print("WITHOUT house data:")
print("-" * 50)
print(response_without['content'])

WITHOUT house data:
--------------------------------------------------
Determining the "most expensive" and "biggest" house in history depends heavily on how value is calculated (market price, estate value at death) and the time period being discussed. However, here are the widely recognized records:

### 🏰 The Most Expensive House
The title generally belongs to **Waddington Hall** in Wiltshire, England.
*   **Value**: It is estimated to be worth between **£1 billion to £1.2 billion** (approximately $1.3 billion to $1.6 billion USD) depending on the valuation source.
*   **Owner**: The late Sir Peter Jones purchased it, and after his death, the family estate value skyrocketed due to the size


In [14]:
# ── Ask WITH context ──────────────────────────────────────────────────────────
# Now the LLM has the actual data — it answers from it
augmented_prompt = generate_prompt(query, house_data)

response_with = generate_with_single_input(
    prompt=augmented_prompt,
    role='user',
    max_tokens=150
)

print("WITH house data:")
print("-" * 50)
print(response_with['content'])

WITH house data:
--------------------------------------------------
Based on the listings provided:

**The most expensive house** is **$750,000** (House 3 at 789 Oak Drive, Capital City, NY 10001).

**The biggest house** is **3,800 sq ft** (also House 3 at 789 Oak Drive, Capital City, NY 10001).


In [15]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
print("QUERY:", query)
print()
print("❌ WITHOUT context (LLM guesses or refuses):")
print(response_without['content'])
print()
print("✅ WITH context (LLM answers from your data):")
print(response_with['content'])

QUERY: What is the most expensive house? And the biggest one?

❌ WITHOUT context (LLM guesses or refuses):
Determining the "most expensive" and "biggest" house in history depends heavily on how value is calculated (market price, estate value at death) and the time period being discussed. However, here are the widely recognized records:

### 🏰 The Most Expensive House
The title generally belongs to **Waddington Hall** in Wiltshire, England.
*   **Value**: It is estimated to be worth between **£1 billion to £1.2 billion** (approximately $1.3 billion to $1.6 billion USD) depending on the valuation source.
*   **Owner**: The late Sir Peter Jones purchased it, and after his death, the family estate value skyrocketed due to the size

✅ WITH context (LLM answers from your data):
Based on the listings provided:

**The most expensive house** is **$750,000** (House 3 at 789 Oak Drive, Capital City, NY 10001).

**The biggest house** is **3,800 sq ft** (also House 3 at 789 Oak Drive, Capital City,

---
<a id='3'></a>
## 3 — Extending the Pattern

<a id='3-1'></a>
### 3.1 Multi-turn conversation with house data

Real applications are conversational — users ask follow-up questions.  
Here the context is passed once in the first message, and follow-ups use `generate_with_multiple_input`.

In [16]:
# ── Conversational RAG with house data ───────────────────────────────────────
# First message includes the data; follow-ups build on the conversation history

first_query = "Which house has the most bedrooms?"
first_prompt = generate_prompt(first_query, house_data)

# Build the conversation starting with the augmented first message
conversation = [
    {'role': 'system', 'content': 'You are a real estate assistant. Answer only from the provided listings.'},
    {'role': 'user',   'content': first_prompt},
]

# Turn 1
response_1 = generate_with_multiple_input(messages=conversation, max_tokens=100)
conversation.append({'role': 'assistant', 'content': response_1['content']})
print("Turn 1 — Q:", first_query)
print("Turn 1 — A:", response_1['content'])
print()

# Turn 2 — follow-up, no need to repeat the house data
followup = "How much does it cost per square foot?"
conversation.append({'role': 'user', 'content': followup})

response_2 = generate_with_multiple_input(messages=conversation, max_tokens=100)
print("Turn 2 — Q:", followup)
print("Turn 2 — A:", response_2['content'])

Turn 1 — Q: Which house has the most bedrooms?
Turn 1 — A: House 3 has the most bedrooms.

Turn 2 — Q: How much does it cost per square foot?
Turn 2 — A: Based on the provided listings, here is the cost per square foot for each house:

*   **House 1 (123 Maple Street):** $153.33 per sq ft ($230,000 / 1,500 sq ft)
*   **House 2 (456 Elm Avenue):** $128.00 per sq ft ($320,000 / 2,500 sq ft


<a id='3-2'></a>
### 3.2 System message + augmented prompt

In production RAG systems, the system message handles the instructions  
and the user message carries the data + question.  
This is cleaner and easier to maintain.

In [17]:
# ── System + user message pattern (production-style) ─────────────────────────
def rag_query(query: str, houses: list) -> str:
    """
    Production-style RAG call:
      - System message:  instructions and persona
      - User message:    data context + question
    """
    context = house_info_layout(houses)
    
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a professional real estate assistant. '
                'Answer only from the house listings provided. '
                'If the answer is not in the listings, say "I do not have that information." '
                'Be concise and factual.'
            )
        },
        {
            'role': 'user',
            'content': f"House listings:\n{context}\nQuestion: {query}"
        }
    ]
    
    response = generate_with_multiple_input(messages=messages, max_tokens=150)
    return response['content']


# Test with different queries
queries = [
    "Which house is best value for money?",
    "Which house was built most recently?",
    "What is the average price across all listings?",
]

for q in queries:
    answer = rag_query(q, house_data)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

Q: Which house is best value for money?
A: The "best value" depends on your specific needs, budget, and preferences (e.g., desired number of bedrooms, location, or year built), as each listing offers a different set of features for its price.

Here is a summary of the options:
*   **House 1:** $230,000, 3 bed / 2 bath, 1,500 sq ft (1998)
*   **House 2:** $320,000, 4 bed / 3 bath, 2,500 sq ft (2005)
*   **House 3:** $750,000, 5 bed

Q: Which house was built most recently?
A: House 3 was built most recently in 2018.

Q: What is the average price across all listings?
A: The average price across all three listings is $433,333.33.

**Calculation:**
*   Total Price: $230,000 + $320,000 + $750,000 = $1,300,000
*   Average: $1,300,000 / 3 = $433,333.33



---
<a id='4'></a>
## 4 — Key Takeaways

Run this cell to print a summary you can refer back to.

In [18]:
summary = """
╔══════════════════════════════════════════════════════════════════╗
║           MODULE 01 LAB 2 — KEY TAKEAWAYS                        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  generate_with_single_input(prompt)                              ║
║    → One message in, one response out                            ║
║    → Use for simple, standalone questions                        ║
║                                                                  ║
║  generate_with_multiple_input(messages)                          ║
║    → Full conversation list in, one response out                 ║
║    → Use for system messages, multi-turn chat                    ║
║                                                                  ║
║  Both return: {'role': 'assistant', 'content': '...'}            ║
║                                                                  ║
║  The augmented prompt pattern:                                   ║
║    1. Format your data as a string                               ║
║    2. Inject it into the prompt with an f-string                 ║
║    3. Ask the question                                           ║
║    4. The LLM answers FROM the data, not from training memory    ║
║                                                                  ║
║  This is the Augment step in: Retrieve → AUGMENT → Generate      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(summary)


╔══════════════════════════════════════════════════════════════════╗
║           MODULE 01 LAB 2 — KEY TAKEAWAYS                        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  generate_with_single_input(prompt)                              ║
║    → One message in, one response out                            ║
║    → Use for simple, standalone questions                        ║
║                                                                  ║
║  generate_with_multiple_input(messages)                          ║
║    → Full conversation list in, one response out                 ║
║    → Use for system messages, multi-turn chat                    ║
║                                                                  ║
║  Both return: {'role': 'assistant', 'content': '...'}            ║
║                                                                  ║
║  The augmented prompt pattern: 

---

*Next: Module 2 — Information Retrieval and Search Foundations*